# COMP8410 A1 - Week 6: Wan2.2 TI2V-5B vs HunyuanVideo 1.5 (image to video) - Colab
**Run the numbered cells in order. Every code cell starts with `# [n.n]` saying what it does and where to run it.**
**Budget: this pair has ~15 units. Phase 1 on a T4 (about 0.2 units), phase 2 on the A100 (about 1 hour).**

| Section | What | Runtime |
|---|---|---|
| 1 | Setup: Drive, repo | any, fresh runtime (1.2 after a kernel restart) |
| 2 | Install + import check + tests + input photos | **T4 first**, repeat on the A100 (about 5 min, no compiling) |
| 3 | Weights (A100 only: 10 GB Wan + Hunyuan on first load) | A100 |
| 4 | Smoke: 1 item per model, then the BUDGET GATE | A100 |
| 5 | Full run (6 items, seed 0), scoring | A100 (scoring can run on the T4) |
| 6 | Report, cost log, disconnect | end |

Before saving: tick *Omit code cell output when saving*. If a cell fails, STOP and read it; do not use Run all. **Disconnect the runtime when finished (6.3).**

## 1. Setup

In [ ]:
# [1.1] SETUP, fresh runtime only: mount Drive, clone the repo, link results/ and outputs/ to Drive
%cd /content
import os
os.environ['HF_HOME'] = '/content/hf'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from google.colab import drive # type: ignore
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs', 'w6_data']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!rm -rf results outputs && ln -s {BASE}/results results && ln -s {BASE}/outputs outputs
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# [1.2] RESTORE after a kernel crash or restart (use INSTEAD of 1.1; the VM files are untouched, only Python variables are lost)
import os
os.environ['HF_HOME'] = '/content/hf'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE = '/content/drive/MyDrive/comp8410_a1'
%cd "/content/a1/48576158 Evaluation"
from google.colab import drive
drive.mount('/content/drive')
!git -C /content/a1 pull -q

## 2. Install, import check, tests, inputs (run on the T4 first; repeat on the A100)

In [ ]:
# [2.1] INSTALL: clone Wan2.2, install requirements without the torch/numpy/transformers pins, diffusers from source, then an import loop that fixes missing modules
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/install_w6.py 2>&1 | tee {BASE}/logs/w6_install.log | tail -25

In [ ]:
# [2.2] CPU TESTS for the W6 scorer + save the pinned environment to Drive (evidence)
%cd "/content/a1/48576158 Evaluation"
!python -m pytest -q tests/test_w6.py
!pip freeze > {BASE}/results/w6_ENV.txt
!cp /content/wan_commit.txt {BASE}/results/w6_commit_wan.txt

In [ ]:
# [2.3] INPUT PHOTOS: find the six listing photos on Drive (W2 copies), crop to 16:9, show the sheet. Same PNGs go to both models.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/prepare_inputs.py --src {BASE} --out /content/w6_work/inputs
from PIL import Image
display(Image.open('/content/w6_work/inputs/inputs_sheet.jpg'))

## 3. Weights (A100 session)

In [ ]:
# [3.1] DOWNLOAD Wan2.2-TI2V-5B (about 10-20 GB). HunyuanVideo-1.5 weights download automatically at the first load in 4.2 (cached in /content/hf).
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/install_w6.py --weights 2>&1 | tail -5
!du -sh /content/ckpt/Wan2.2-TI2V-5B

## 4. Smoke tests and the budget gate

In [ ]:
# [4.1] SMOKE WAN: first item, seed 0 (loads the model, one cheap warm-up, one full 73-frame clip). Writes results/w6_smoke.jsonl, not the real file.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/gen_w6.py wan22 --smoke 2>&1 | tee {BASE}/logs/w6_smoke_wan.log | grep -v Warning | tail -30

In [ ]:
# [4.2] SMOKE HUNYUAN: same, first item. Downloads the weights on first use.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/gen_w6.py hunyuanvideo15 --smoke 2>&1 | tee {BASE}/logs/w6_smoke_hunyuan.log | grep -v Warning | tail -30

In [ ]:
# [4.3] BUDGET GATE: projects units for the full run from the smoke latencies. STOP if the projection is over the pair budget.
import json
UNITS_PER_HR = 11.77      # A100 on the Resources panel: check and edit
PAIR_BUDGET = 15.0        # units for the rest of this pair
rows = [json.loads(l) for l in open('results/w6_smoke.jsonl')]
lat = {r['model']: r['value'] for r in rows if r['metric'] == 'latency_s' and r['item_id'] != 'all'}
load = {r['model']: r['value'] for r in rows if r['metric'] == 'load_time_s'}
n = 6
est_s = sum(load.values()) + n * sum(lat.values())
print('latency per clip (s):', lat, '| load (s):', load)
print(f'projected full run: {est_s/60:.0f} min = {est_s/3600*UNITS_PER_HR:.1f} units (budget {PAIR_BUDGET})')
print('GATE', 'PASSED' if est_s / 3600 * UNITS_PER_HR <= PAIR_BUDGET * 0.7 else 'FAILED: cut the item count or frames before running 5.x')

In [ ]:
# [4.4] LOOK AT THE SMOKE CLIPS (first frame vs last frame of each)
%cd "/content/a1/48576158 Evaluation"
import glob, numpy as np
from PIL import Image
import sys; sys.path.insert(0, 'pairs/w6_video_generation')
from score_w6 import read_video
for p in sorted(glob.glob('outputs/w6_smoke/*/*.mp4')):
    if '_warmup' in p: continue
    fr, fps = read_video(__import__('pathlib').Path(p))
    print(p, len(fr), 'frames', fps, 'fps', fr[0].shape)
    display(Image.fromarray(np.concatenate([np.asarray(Image.fromarray(fr[i]).resize((320, 180))) for i in (0, len(fr)//2, -1)], axis=1)))

## 5. Full run and scoring (seed 0, 6 items). Resumable: re-running skips finished clips.

In [ ]:
# [5.1] FULL RUN: Wan2.2 (all items)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/gen_w6.py wan22 2>&1 | tee {BASE}/logs/w6_full_wan.log | grep -v Warning | tail -20

In [ ]:
# [5.2] FULL RUN: HunyuanVideo 1.5 (all items)
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/gen_w6.py hunyuanvideo15 2>&1 | tee {BASE}/logs/w6_full_hunyuan.log | grep -v Warning | tail -20

In [ ]:
# [5.3] SCORE all clips (CPU is fine, GPU faster). Add --rescore to recompute.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/score_w6.py 2>&1 | tee {BASE}/logs/w6_score.log | grep -v Warning | tail -20

## 6. Results, cost, shut down

In [ ]:
# [6.1] TABLES, CHARTS, FRAME STRIPS; copies report assets to Drive. The strips show the rooms: keep 'Omit code cell output' ticked.
%cd "/content/a1/48576158 Evaluation"
!python pairs/w6_video_generation/report_w6.py --show 2>&1 | tee {BASE}/logs/w6_report.log | grep -v Warning
!mkdir -p {BASE}/report_assets && cp -r report_assets/w6 {BASE}/report_assets/
from PIL import Image
display(Image.open('outputs/w6_figures/w6_strips.jpg'))

In [ ]:
# [6.2] COST LOG: fill in HOURS (connected GPU time) and UNITS_PER_HR (Resources panel), run once per session
%cd "/content/a1/48576158 Evaluation"
HOURS = None          # e.g. 1.2
UNITS_PER_HR = None   # e.g. 11.77
if HOURS and UNITS_PER_HR:
    from a1eval import log_cost
    log_cost('cost_log.csv', '2026-11-02', 'Colab Pro', 'A100-40GB', HOURS, UNITS_PER_HR * 0.1513, note='W6 install + generation + scoring')
    !tail -3 cost_log.csv

In [ ]:
# [6.3] FREE THE GPU: set AUTO_DISCONNECT = True once everything is copied to Drive
AUTO_DISCONNECT = False
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()